# LSTM / BiLSTM on UCI HAR — Nandan Reddy Maram

This notebook owns the recurrent model family for the ICT-4442 HAR comparison. It follows the
shared protocol in `README.md`:

- loads only `processed/sequence.npz` through `har_common` (no re-splitting, no re-fitting);
- baseline: unidirectional LSTM, constant LR, no augmentation, no class weighting;
- tuned: small architecture + recipe search, warm-up + cosine LR, pre-augmented train set,
  dropout, weight decay, label smoothing and optional EMA;
- model selection and early stopping on **validation macro-F1**, best-epoch weights restored;
- the test set is evaluated once for the selected tuned model (and once for each fixed baseline
  used in the before/after table);
- every checklist metric and artefact is written under `lstm/` via `hc.save_results`.

Set `QUICK_TEST = True` only for an end-to-end smoke test (it writes to `lstm/_quick_test/`).
Set `HAR_PREVIEW=1` (or `PREVIEW = True`) for a reduced CPU run that populates `lstm/` with
preview artifacts; these are clearly marked and are **not** for the report. For final results,
use a T4 GPU and leave both flags `False`, then **Run all**.


## 0. Setup and project paths

Mounts Drive when in Colab, locates the project folder, imports `har_common`, and creates the output folders.

In [ ]:
# ---------------------------------------------------------------- 0. Setup
import os, sys, json, time, copy, math, hashlib, itertools, warnings, random, gc, contextlib, shutil
from pathlib import Path
import numpy as np
import pandas as pd

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    drive.mount("/content/drive")

import matplotlib
if not IN_COLAB:
    matplotlib.use("Agg")
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import f1_score, classification_report

warnings.filterwarnings("ignore", category=UserWarning)


def find_project_root():
    """Locate 'DL PROJ' on Drive, locally, or via HAR_PROJECT_ROOT."""
    env = os.environ.get("HAR_PROJECT_ROOT")
    if env:
        return Path(env)
    starts = [Path("/content/drive/MyDrive"), Path.cwd(), *Path.cwd().parents]
    seen = set()
    for start in starts:
        try:
            if not start.exists() or start in seen:
                continue
            seen.add(start)
        except OSError:
            continue
        if start.is_dir() and start.name.lower().replace("_", " ") == "dl proj":
            return start
        try:
            for p in [start, *start.iterdir()]:
                if p.is_dir() and p.name.lower().replace("_", " ") == "dl proj":
                    return p
        except (PermissionError, OSError):
            pass
        try:
            for p in start.iterdir():
                if not p.is_dir():
                    continue
                for q in p.iterdir():
                    if q.is_dir() and q.name.lower().replace("_", " ") == "dl proj":
                        return q
        except (PermissionError, OSError):
            pass
    raise FileNotFoundError(
        "Could not find the 'DL PROJ' folder. In Colab mount Drive first, or set "
        "os.environ['HAR_PROJECT_ROOT'] to the project path."
    )


## 1. Run settings

Shared-protocol knobs. `QUICK_TEST` uses a tiny subset and 2 epochs. `PREVIEW` uses the full
data but a reduced grid and 12 epochs, so the folder can be populated on CPU; keep both
`False` for the report.

In [ ]:
# ---------------------------------------------------------------- 1. Run settings
QUICK_TEST = os.environ.get("HAR_QUICK_TEST", "0") == "1"
PREVIEW = (not QUICK_TEST) and os.environ.get("HAR_PREVIEW", "0") == "1"

EPOCHS = 200                  # tuned runs: max epochs
PATIENCE = 25                 # tuned runs: early stopping on validation macro-F1
BATCH_SIZE = 64               # shared protocol
MAX_CPU_LATENCY_MS = 10.0     # deployment budget; None disables the filter
RUN_BASELINES = True          # LSTM and parameter-matched BiLSTM baselines
RUN_STAGE_A = True            # architecture search
RUN_STAGE_B = True            # optimiser / LR / dropout search
RUN_ABLATIONS = True          # validation-only ablation study

if QUICK_TEST:
    EPOCHS, PATIENCE = 2, 2
    MAX_CPU_LATENCY_MS = None

if PREVIEW:
    EPOCHS, PATIENCE = 12, 4

PROJECT_ROOT = find_project_root()
PROCESSED_DIR = PROJECT_ROOT / "processed"
OUT = PROJECT_ROOT / "lstm"
if QUICK_TEST:
    OUT = OUT / "_quick_test"
RUNS_DIR, FIG_DIR, FINAL_DIR = OUT / "runs", OUT / "figures", OUT / "final_model"
for _d in (RUNS_DIR, FIG_DIR, FINAL_DIR):
    _d.mkdir(parents=True, exist_ok=True)

# A final run removes preview run directories and the preview warning.
if not PREVIEW and not QUICK_TEST:
    for _d in RUNS_DIR.glob("*"):
        _rj = _d / "result.json"
        if _rj.exists():
            try:
                if json.loads(_rj.read_text(encoding="utf-8")).get("preview"):
                    shutil.rmtree(_d)
            except Exception:
                pass
    _p = OUT / "PREVIEW_RUN.txt"
    if _p.exists():
        _p.unlink()

if PREVIEW:
    (OUT / "PREVIEW_RUN.txt").write_text(
        "PREVIEW RUN ONLY - NOT FINAL RESULTS\n"
        f"Reduced CPU run: {EPOCHS} epochs, patience {PATIENCE}, reduced grids and ablations.\n"
        "For final results, set HAR_PREVIEW=0 (or PREVIEW=False) and run on a T4 GPU.\n",
        encoding="utf-8",
    )

sys.path.insert(0, str(PROJECT_ROOT))
import har_common as hc

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SEED = 42
OWNER = "Nandan Reddy Maram"

print("Project root :", PROJECT_ROOT)
print("Outputs      :", OUT)
print("Device       :", DEVICE, torch.cuda.get_device_name(0) if DEVICE == "cuda" else "")
print("PyTorch      :", torch.__version__)
print("har_common   :", hc.__file__)
if DEVICE == "cpu":
    print("WARNING: no GPU. Runtime -> Change runtime type -> T4 GPU (CPU will be slow).")


## 2. Load the shared, preprocessed UCI HAR data

Loads the shared sequence representation `(N, 128, 9)` and the optional pre-augmented train set. Labels stay in canonical order 0-5.

In [ ]:
# ---------------------------------------------------------------- 2. Data
data = hc.load_data("sequence", processed_dir=PROCESSED_DIR)
data_aug = hc.load_data("sequence", augmented=True, processed_dir=PROCESSED_DIR)
meta = data["meta"]


def _subset(dd, n_train=512, n_val=256, n_test=512):
    out = dict(dd)
    for split, n in (("train", n_train), ("val", n_val), ("test", n_test)):
        for key in (f"X_{split}", f"y_{split}", f"subj_{split}"):
            if key in dd:
                out[key] = dd[key][:n]
    return out


if QUICK_TEST:
    data = _subset(data)
    data_aug = _subset(data_aug)

print("dataset_version:", meta["dataset_version"])
for s in ("train", "val", "test"):
    print(f"{s:5s}", data[f"X_{s}"].shape, np.bincount(data[f"y_{s}"], minlength=6).tolist())
print("augmented train:", data_aug["X_train"].shape)

to_t = lambda a, dt=torch.float32: torch.as_tensor(a, dtype=dt, device=DEVICE)
Xtr, ytr = to_t(data["X_train"]), to_t(data["y_train"], torch.long)
Xva, yva = to_t(data["X_val"]), to_t(data["y_val"], torch.long)
Xtr_aug, ytr_aug = to_t(data_aug["X_train"]), to_t(data_aug["y_train"], torch.long)
# Test stays on the CPU until the single final evaluation.


## 3. LSTM / BiLSTM model

The recurrent encoder: stacked (Bi)LSTM, last/mean/max temporal pooling, LayerNorm, dropout, linear head. `analytic_flops` supplies the RNN gate FLOPs.

In [ ]:
# ---------------------------------------------------------------- 3. Model
class HARLSTM(nn.Module):
    """(Bi)LSTM -> LayerNorm -> Dropout -> Linear, for (B, 128, 9) windows."""

    def __init__(self, hidden_size=128, num_layers=1, dropout=0.3, bidirectional=False,
                 pooling="last", input_size=9, n_classes=6, **_):
        super().__init__()
        self.input_size = int(input_size)
        self.hidden_size = int(hidden_size)
        self.num_layers = int(num_layers)
        self.bidirectional = bool(bidirectional)
        self.pooling = pooling
        self.n_classes = int(n_classes)
        self.n_directions = 2 if self.bidirectional else 1

        self.lstm = nn.LSTM(
            input_size=self.input_size,
            hidden_size=self.hidden_size,
            num_layers=self.num_layers,
            batch_first=True,
            bidirectional=self.bidirectional,
            dropout=float(dropout) if self.num_layers > 1 else 0.0,
        )
        feat = self.hidden_size * self.n_directions
        self.norm = nn.LayerNorm(feat)
        self.drop = nn.Dropout(float(dropout))
        self.fc = nn.Linear(feat, self.n_classes)
        self.apply(self._init)

    @staticmethod
    def _init(m):
        if isinstance(m, nn.Linear):
            nn.init.xavier_uniform_(m.weight)
            if m.bias is not None:
                nn.init.zeros_(m.bias)

    def forward(self, x):
        out, (h_n, _) = self.lstm(x)
        if self.pooling == "last":
            z = torch.cat([h_n[-2], h_n[-1]], dim=-1) if self.bidirectional else h_n[-1]
        elif self.pooling == "mean":
            z = out.mean(dim=1)
        elif self.pooling == "max":
            z = out.max(dim=1).values
        else:
            raise ValueError(f"Unknown pooling: {self.pooling}")
        return self.fc(self.drop(self.norm(z)))

    def analytic_flops(self, seq_len=128):
        """Gate matmul FLOPs (2 per multiply-add); used when flop_counter skips RNN ops."""
        total = 0.0
        for layer in range(self.num_layers):
            in_sz = self.input_size if layer == 0 else self.hidden_size * self.n_directions
            per_direction = 2 * 4 * seq_len * (in_sz * self.hidden_size + self.hidden_size ** 2)
            total += per_direction * self.n_directions
        total += 2 * self.hidden_size * self.n_directions * self.n_classes
        return float(total)


## 4. Training utilities (metrics, EMA, optimisers, schedules, FLOPs)

Reusable helpers: probability inference, plain-CE evaluation, EMA, AdamW/Adam/RMSprop/SGD, warm-up + cosine schedule, AMP scaler and FLOP counting.

In [ ]:
# ---------------------------------------------------------------- 4. Training utilities
class EMA:
    """Exponential moving average of all floating-point weights and buffers."""

    def __init__(self, model, decay):
        self.module = copy.deepcopy(model).eval()
        for p in self.module.parameters():
            p.requires_grad_(False)
        self.decay, self.n = float(decay), 0

    @torch.no_grad()
    def update(self, model):
        self.n += 1
        d = min(self.decay, (1 + self.n) / (10 + self.n))
        for e, m in zip(self.module.state_dict().values(), model.state_dict().values()):
            if e.dtype.is_floating_point:
                e.mul_(d).add_(m.detach(), alpha=1 - d)
            else:
                e.copy_(m)


def param_groups(model, wd):
    decay, no_decay = [], []
    for n, p in model.named_parameters():
        if not p.requires_grad:
            continue
        (no_decay if p.ndim <= 1 else decay).append(p)
    return [{"params": decay, "weight_decay": float(wd)},
            {"params": no_decay, "weight_decay": 0.0}]


def make_optimizer(model, cfg):
    name = cfg["optimizer"].lower()
    if name == "adamw":
        return torch.optim.AdamW(param_groups(model, cfg["weight_decay"]), lr=cfg["lr"],
                                 betas=(0.9, 0.999))
    if name == "adam":
        return torch.optim.Adam(param_groups(model, cfg["weight_decay"]), lr=cfg["lr"],
                                betas=(0.9, 0.999))
    if name == "rmsprop":
        return torch.optim.RMSprop(model.parameters(), lr=cfg["lr"], momentum=0.9,
                                   weight_decay=cfg["weight_decay"])
    if name == "sgd":
        return torch.optim.SGD(model.parameters(), lr=cfg["lr"], momentum=0.9,
                               weight_decay=cfg["weight_decay"], nesterov=True)
    raise ValueError(f"Unknown optimizer: {cfg['optimizer']}")


def make_lr_lambda(cfg, steps_per_epoch):
    warm = cfg["warmup_epochs"] * steps_per_epoch
    total = cfg["epochs"] * steps_per_epoch
    floor = cfg["min_lr"] / cfg["lr"] if cfg["lr"] > 0 else 0.0

    def f(step):
        if cfg["schedule"] == "constant":
            return 1.0
        if warm > 0 and step < warm:
            return (step + 1) / warm
        t = min((step - warm) / max(total - warm, 1), 1.0)
        return floor + (1 - floor) * 0.5 * (1 + math.cos(math.pi * t))

    return f


def autocast_ctx():
    if DEVICE == "cuda":
        return torch.autocast(device_type="cuda", dtype=torch.float16)
    return contextlib.nullcontext()


def make_scaler():
    try:
        return torch.amp.GradScaler("cuda", enabled=DEVICE == "cuda")
    except (AttributeError, TypeError):
        return torch.cuda.amp.GradScaler(enabled=DEVICE == "cuda")


@torch.no_grad()
def predict_proba(model, X, batch=1024):
    """Softmax probabilities for numpy or tensor input (N, 128, 9)."""
    model.eval()
    dev = next(model.parameters()).device
    out = []
    for i in range(0, len(X), batch):
        xb = torch.as_tensor(X[i:i + batch], dtype=torch.float32, device=dev)
        out.append(F.softmax(model(xb).float(), -1).cpu())
    return torch.cat(out).numpy()


@torch.no_grad()
def eval_split(model, X, y):
    p = predict_proba(model, X)
    yt = y.cpu().numpy() if torch.is_tensor(y) else np.asarray(y)
    pred = p.argmax(1)
    loss = float(-np.log(np.clip(p[np.arange(len(yt)), yt], 1e-12, None)).mean())
    return loss, float((pred == yt).mean()), float(f1_score(yt, pred, average="macro", zero_division=0)), p


def roughness(series):
    s = np.asarray(series, dtype=float)
    return float(np.abs(np.diff(s)).mean()) if len(s) > 1 else float("nan")


def cfg_id(cfg):
    return hashlib.md5(json.dumps(cfg, sort_keys=True).encode()).hexdigest()[:8]


def count_flops_lstm(model, input_shape=(1, 128, 9)):
    """Analytical LSTM gate FLOPs; torch's counter usually counts only the linear head."""
    analytic = float(model.analytic_flops(seq_len=input_shape[1]))
    try:
        f = hc.count_flops(model, input_shape)
    except Exception:
        f = None
    return max(analytic, float(f) if f else 0.0)


def plot_history(h, best_epoch, title, path=None, show=True):
    fig, ax = plt.subplots(1, 4, figsize=(22, 4.6))
    ax[0].plot(h.epoch, h.train_loss, label="train (clean)")
    ax[0].plot(h.epoch, h.val_loss, label="validation")
    if "train_loss_optimised" in h.columns:
        ax[0].plot(h.epoch, h.train_loss_optimised, ":", alpha=.7,
                   label="train (as optimised)")
    ax[0].set_title("Cross-entropy loss")
    ax[0].set_yscale("log")
    ax[1].plot(h.epoch, h.train_acc, label="train")
    ax[1].plot(h.epoch, h.val_acc, label="validation")
    ax[1].set_title("Accuracy")
    ax[2].plot(h.epoch, h.train_macro_f1, label="train")
    ax[2].plot(h.epoch, h.val_macro_f1, label="validation")
    ax[2].set_title("Macro-F1")
    ax[3].plot(h.epoch, h.lr, color="tab:purple")
    ax[3].set_title("Learning rate")
    ax[3].set_yscale("log")
    for a in ax:
        a.axvline(best_epoch, ls="--", c="gray", lw=1)
        a.set_xlabel("epoch")
        a.grid(alpha=.3)
    for a in ax[:3]:
        a.legend()
    fig.suptitle(title)
    fig.tight_layout()
    if path:
        fig.savefig(path, dpi=150, bbox_inches="tight")
    if show:
        plt.show()
    plt.close(fig)


## 5. One training run (validation macro-F1 model selection)

One self-contained training run: mixed precision, gradient clipping, early stopping on validation macro-F1, best-epoch restore, per-run history and result JSON.

In [ ]:
# ---------------------------------------------------------------- 5. One training run
def train_run(name, cfg, verbose=True):
    """Train one configuration; reuse a finished run with the same config hash."""
    run_dir = RUNS_DIR / f"{name}_{cfg_id(cfg)}"
    if (run_dir / "result.json").exists():
        res = json.load(open(run_dir / "result.json"))
        print(f"[skip] {name}: already done (val macro-F1 {res['val_macro_f1']:.4f})")
        return res
    run_dir.mkdir(parents=True, exist_ok=True)
    json.dump(cfg, open(run_dir / "config.json", "w"), indent=2)

    random.seed(cfg["seed"])
    np.random.seed(cfg["seed"])
    torch.manual_seed(cfg["seed"])
    if DEVICE == "cuda":
        torch.cuda.manual_seed_all(cfg["seed"])
        torch.cuda.reset_peak_memory_stats()
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

    model = HARLSTM(**cfg).to(DEVICE)
    ema = EMA(model, cfg["ema_decay"]) if cfg.get("ema") else None
    opt = make_optimizer(model, cfg)

    X_fit = Xtr_aug if cfg.get("augmented") else Xtr
    y_fit = ytr_aug if cfg.get("augmented") else ytr
    n, bs = len(X_fit), cfg["batch_size"]
    steps = max(n // bs, 1)
    sched = torch.optim.lr_scheduler.LambdaLR(opt, make_lr_lambda(cfg, steps))
    scaler_amp = make_scaler()
    gen = torch.Generator(device="cpu").manual_seed(cfg["seed"])

    hist, best = [], dict(f1=-1.0, loss=1e9, epoch=0, state=None)
    t0 = time.time()
    for epoch in range(1, cfg["epochs"] + 1):
        te = time.time()
        model.train()
        perm = torch.randperm(n, generator=gen).to(DEVICE)
        run_loss = 0.0
        for i in range(steps):
            idx = perm[i * bs:(i + 1) * bs]
            xb, yb = X_fit[idx], y_fit[idx]
            with autocast_ctx():
                loss = F.cross_entropy(model(xb), yb,
                                       label_smoothing=float(cfg.get("label_smoothing", 0.0)))
            opt.zero_grad(set_to_none=True)
            scaler_amp.scale(loss).backward()
            scaler_amp.unscale_(opt)
            nn.utils.clip_grad_norm_(model.parameters(), cfg["grad_clip"])
            scaler_amp.step(opt)
            scaler_amp.update()
            sched.step()
            if ema:
                ema.update(model)
            run_loss += loss.item()

        ev = ema.module if ema else model
        tr_loss, tr_acc, tr_f1, _ = eval_split(ev, Xtr, ytr)
        va_loss, va_acc, va_f1, _ = eval_split(ev, Xva, yva)
        hist.append(dict(
            epoch=epoch,
            train_loss=tr_loss,
            train_acc=tr_acc,
            train_macro_f1=tr_f1,
            val_loss=va_loss,
            val_acc=va_acc,
            val_macro_f1=va_f1,
            lr=opt.param_groups[0]["lr"],
            epoch_time_s=time.time() - te,
            train_loss_optimised=run_loss / steps,
        ))

        improved = (va_f1 > best["f1"] + 1e-4 or
                    (abs(va_f1 - best["f1"]) <= 1e-4 and va_loss < best["loss"]))
        if improved:
            best = dict(f1=va_f1, loss=va_loss, epoch=epoch, acc=va_acc,
                        state={k: v.detach().cpu().clone() for k, v in ev.state_dict().items()})

        if verbose and (epoch == 1 or epoch % 10 == 0 or (improved and epoch > 10)):
            print(f"  {name} ep {epoch:3d} | train loss {tr_loss:.4f} acc {tr_acc:.4f} | "
                  f"val loss {va_loss:.4f} acc {va_acc:.4f} F1 {va_f1:.4f} | "
                  f"lr {hist[-1]['lr']:.2e}{'  *' if improved else ''}")

        if epoch - best["epoch"] >= cfg["patience"]:
            break

    train_time = time.time() - t0
    peak_mem = torch.cuda.max_memory_allocated() / 2 ** 20 if DEVICE == "cuda" else float("nan")

    torch.save({"state_dict": best["state"], "config": cfg}, run_dir / "best.pt")
    h = pd.DataFrame(hist)
    h.to_csv(run_dir / "history.csv", index=False)
    model.load_state_dict(best["state"])
    lat = hc.measure_latency(model, (1, 128, 9), "cpu", n_warmup=20, n_runs=100)

    res = dict(
        name=name, run_id=cfg_id(cfg), run_dir=str(run_dir), preview=PREVIEW,
        **{f"cfg_{k}": v for k, v in cfg.items()},
        val_macro_f1=best["f1"], val_acc=best["acc"], val_loss_at_best=best["loss"],
        best_epoch=best["epoch"], epochs_trained=len(h), train_time_s=train_time,
        time_per_epoch_s=train_time / len(h), peak_gpu_mem_mb=peak_mem,
        params_total=hc.count_params(model), latency_ms_cpu_b1=lat,
        train_macro_f1_at_best=float(h.loc[best["epoch"] - 1, "train_macro_f1"]),
        val_loss_roughness=roughness(h["val_loss"]),
        val_f1_roughness=roughness(h["val_macro_f1"]),
        min_val_loss=float(h["val_loss"].min()),
    )
    json.dump(res, open(run_dir / "result.json", "w"), indent=2, default=float)
    plot_history(h, best["epoch"], f"{name} (val macro-F1 {best['f1']:.4f})",
                 run_dir / "curves.png", show=False)
    print(f"[done] {name}: val macro-F1 {best['f1']:.4f} acc {best['acc']:.4f} "
          f"@ epoch {best['epoch']} ({len(h)} epochs, {train_time:.0f}s, "
          f"{res['params_total']:,} params, CPU {lat:.2f} ms)")
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
    return res


def record(res, stage):
    """Append/refresh one row in grid_search_results.csv."""
    path = OUT / "grid_search_results.csv"
    row = pd.DataFrame([dict(stage=stage, **res)])
    if path.exists():
        old = pd.read_csv(path)
        if "preview" in old.columns:
            old = old[old["preview"] == PREVIEW]
        old = old[~((old["run_id"] == res["run_id"]) & (old["name"] == res["name"]))]
        row = pd.concat([old, row], ignore_index=True)
    row.to_csv(path, index=False)


def grid(base, space, prefix):
    keys = list(space)
    short = dict(bidirectional="bi", hidden_size="h", num_layers="L", pooling="pool",
                 optimizer="opt", lr="lr", dropout="do", augmented="aug",
                 weight_decay="wd", label_smoothing="ls", schedule="sched", ema="ema")

    def fmt(v):
        if isinstance(v, bool):
            return "T" if v else "F"
        if isinstance(v, float):
            return f"{v:g}"
        return str(v)

    for vals in itertools.product(*space.values()):
        cfg = dict(base, **dict(zip(keys, vals)))
        tag = "_".join(f"{short.get(k, k)}{fmt(v)}" for k, v in zip(keys, vals))
        yield f"{prefix}_{tag}", cfg


def pick_best(results):
    ok = [r for r in results
          if MAX_CPU_LATENCY_MS is None or r["latency_ms_cpu_b1"] <= MAX_CPU_LATENCY_MS]
    if not ok:
        print("No run met the latency budget - selecting among all runs.")
        ok = results
    return max(ok, key=lambda r: (round(r["val_macro_f1"], 4), -r["val_loss_at_best"]))


def cfg_from(res):
    return {k[4:]: v for k, v in res.items() if k.startswith("cfg_")}


def model_name_for(cfg):
    return "bilstm" if cfg["bidirectional"] else "lstm"


def notes_for(cfg, res, tag):
    arch = (f"{'Bi' if cfg['bidirectional'] else ''}LSTM hidden {cfg['hidden_size']} x "
            f"{cfg['num_layers']} layer(s), pooling {cfg['pooling']}, dropout {cfg['dropout']}")
    aug = "pre-augmented shared train set" if cfg["augmented"] else "no augmentation"
    prefix = "PREVIEW (reduced CPU run; NOT final) - " if PREVIEW else ""
    return (prefix + f"{tag}: {arch}; {cfg['optimizer']} lr {cfg['lr']:g} wd {cfg['weight_decay']:g}, "
            f"{cfg['schedule']} schedule (warm-up {cfg['warmup_epochs']} ep, min lr "
            f"{cfg['min_lr']:g}), label smoothing {cfg['label_smoothing']:g}, "
            f"EMA {'on' if cfg['ema'] else 'off'}, {aug}, batch {cfg['batch_size']}, "
            f"AMP, grad-clip {cfg['grad_clip']}, early stop patience {cfg['patience']} / "
            f"max {cfg['epochs']} ep, best epoch {res['best_epoch']}. "
            f"FLOPs: analytical LSTM gate count (2 per multiply-add; torch flop_counter does not "
            f"count RNN ops); CPU latency on 1 thread.")


def training_for(res):
    return {k: res[k] for k in ("train_time_s", "epochs_trained", "best_epoch", "time_per_epoch_s")}


def efficiency_for(model, weights_path, res):
    return dict(
        params_total=hc.count_params(model),
        model_size_mb=hc.file_size_mb(weights_path),
        flops_per_window=count_flops_lstm(model, (1, 128, 9)),
        latency_ms_cpu_b1=hc.measure_latency(model, (1, 128, 9), "cpu"),
        latency_ms_gpu_b1=hc.measure_latency(model, (1, 128, 9), "cuda") if DEVICE == "cuda" else np.nan,
        throughput_gpu_wps=hc.measure_throughput(model, (128, 9), 256, "cuda") if DEVICE == "cuda" else np.nan,
        peak_gpu_mem_mb=res.get("peak_gpu_mem_mb", np.nan),
    )


def load_run_model(res):
    ckpt = torch.load(Path(res["run_dir"]) / "best.pt", map_location="cpu", weights_only=False)
    model = HARLSTM(**ckpt["config"]).to(DEVICE)
    model.load_state_dict(ckpt["state_dict"])
    model.eval()
    return model, ckpt


## 6. Baseline and tuning configurations

Baseline configs and the two search grids. The baseline BiLSTM uses hidden size 88 per direction to stay within about 2% of the 128-unit LSTM parameter count.

In [ ]:
# ---------------------------------------------------------------- 6. Configurations
BASE_CFG = dict(
    hidden_size=128, num_layers=1, dropout=0.3, bidirectional=True, pooling="last",
    optimizer="adamw", lr=1e-3, weight_decay=0.01, schedule="cosine", warmup_epochs=10,
    min_lr=1e-5, grad_clip=1.0, label_smoothing=0.05, ema=True, ema_decay=0.998,
    augmented=True, batch_size=BATCH_SIZE, epochs=EPOCHS, patience=PATIENCE, seed=SEED,
)

BASELINE_LSTM_CFG = dict(
    BASE_CFG, hidden_size=128, num_layers=1, dropout=0.3, bidirectional=False, pooling="last",
    optimizer="adamw", lr=1e-3, weight_decay=0.01, schedule="constant", warmup_epochs=0,
    min_lr=1e-3, label_smoothing=0.0, ema=False, augmented=False,
    epochs=min(150, EPOCHS), patience=min(20, PATIENCE),
)
# Hidden 88 x 2 directions is within ~2% of the 128-unit unidirectional LSTM parameter count.
BASELINE_BILSTM_CFG = dict(BASELINE_LSTM_CFG, hidden_size=88, bidirectional=True)

STAGE_A_GRID = dict(bidirectional=[False, True], hidden_size=[64, 128], num_layers=[1, 2])
STAGE_B_GRID = dict(optimizer=["adamw", "adam"], lr=[5e-4, 1e-3], dropout=[0.2, 0.4])

if QUICK_TEST:
    BASE_CFG.update(epochs=EPOCHS, patience=PATIENCE, warmup_epochs=1)
    BASELINE_LSTM_CFG.update(epochs=EPOCHS, patience=PATIENCE)
    BASELINE_BILSTM_CFG.update(epochs=EPOCHS, patience=PATIENCE)
    STAGE_A_GRID = dict(bidirectional=[False, True], hidden_size=[64], num_layers=[1])
    STAGE_B_GRID = dict(optimizer=["adamw", "adam"], lr=[1e-3], dropout=[0.3])

if PREVIEW:
    BASE_CFG.update(warmup_epochs=2, min_lr=1e-4)
    STAGE_A_GRID = dict(bidirectional=[False, True], hidden_size=[64, 128], num_layers=[1])
    STAGE_B_GRID = dict(optimizer=["adamw", "adam"], lr=[1e-3], dropout=[0.3])


## 7. Baseline runs

Trains the two fixed baselines. Both are logged in `lstm_results.csv` with `run_tag='baseline'`.

In [ ]:
# ---------------------------------------------------------------- 7. Baseline runs
baselines = []
if RUN_BASELINES:
    baseline_specs = [
        ("baseline_lstm", BASELINE_LSTM_CFG, "baseline LSTM"),
        ("baseline_bilstm", BASELINE_BILSTM_CFG, "parameter-matched baseline BiLSTM"),
    ]
    for name, cfg, label in baseline_specs:
        print("\n" + "=" * 70)
        print(label)
        print("=" * 70)
        r = train_run(name, cfg)
        record(r, "baseline")
        baselines.append(r)
    print("\nBaselines:")
    print(pd.DataFrame(baselines)[["name", "cfg_hidden_size", "cfg_bidirectional", "params_total",
                                   "val_macro_f1", "val_acc", "best_epoch", "epochs_trained"]]
          .sort_values("val_macro_f1", ascending=False).to_string(index=False))


## 8. Stage A: architecture search

Searches LSTM vs BiLSTM, hidden size 64/128 and depth 1/2 using the tuned recipe. Selection is on validation macro-F1 with a CPU-latency budget.

In [ ]:
# ---------------------------------------------------------------- 8. Stage A - architecture search
stage_a = []
best_a = None
if RUN_STAGE_A:
    print("\n" + "=" * 70)
    print("STAGE A - architecture search")
    print("=" * 70)
    for name, cfg in grid(BASE_CFG, STAGE_A_GRID, "A"):
        r = train_run(name, cfg)
        record(r, "A_architecture")
        stage_a.append(r)
    best_a = pick_best(stage_a)
    print("\nStage A winner:", best_a["name"], f"val macro-F1 {best_a['val_macro_f1']:.4f}")
    print(pd.DataFrame(stage_a)[["name", "cfg_bidirectional", "cfg_hidden_size", "cfg_num_layers",
                                 "params_total", "val_macro_f1", "val_acc", "best_epoch",
                                 "epochs_trained", "latency_ms_cpu_b1"]]
          .sort_values("val_macro_f1", ascending=False).to_string(index=False))


## 9. Stage B: optimiser, learning-rate and dropout search

Searches AdamW vs Adam, peak LR and dropout around the Stage A winner.

In [ ]:
# ---------------------------------------------------------------- 9. Stage B - recipe search
stage_b = []
if RUN_STAGE_B:
    print("\n" + "=" * 70)
    print("STAGE B - optimiser / LR / dropout search")
    print("=" * 70)
    if best_a is not None:
        arch = {k: cfg_from(best_a)[k] for k in STAGE_A_GRID}
    else:
        arch = {k: BASE_CFG[k] for k in STAGE_A_GRID}
    for name, cfg in grid(dict(BASE_CFG, **arch), STAGE_B_GRID, "B"):
        r = train_run(name, cfg)
        record(r, "B_recipe")
        stage_b.append(r)
    print(pd.DataFrame(stage_b)[["name", "cfg_optimizer", "cfg_lr", "cfg_dropout",
                                 "val_macro_f1", "val_acc", "best_epoch", "epochs_trained"]]
          .sort_values("val_macro_f1", ascending=False).to_string(index=False))

search_results = stage_a + stage_b
if not search_results:
    search_results = baselines
if not search_results:
    raise RuntimeError("No runs were executed. Enable baselines or at least one search stage.")
best = pick_best(search_results)
BEST_CFG = cfg_from(best)
print("\nSELECTED TUNED CONFIGURATION:", best["name"],
      f"(val macro-F1 {best['val_macro_f1']:.4f})")
print(json.dumps(BEST_CFG, indent=2))
json.dump({"selected_run": best["name"], "config": BEST_CFG, "preview": PREVIEW,
           "val_macro_f1": best["val_macro_f1"], "val_acc": best["val_acc"]},
          open(OUT / "best_config.json", "w"), indent=2, default=float)


## 10. Ablations (validation only)

Ablates augmentation, the LR schedule, AdamW, EMA, label smoothing and dropout. These runs are validation-only.

In [ ]:
# ---------------------------------------------------------------- 10. Ablations (validation only)
ablation = []
if RUN_ABLATIONS:
    print("\n" + "=" * 70)
    print("ABLATIONS - validation only")
    print("=" * 70)
    variants = {
        "abl_no_augmentation": dict(BEST_CFG, augmented=False),
        "abl_constant_lr": dict(BEST_CFG, schedule="constant", warmup_epochs=0),
        "abl_adam_instead_of_adamw": dict(BEST_CFG, optimizer="adam"),
        "abl_no_ema": dict(BEST_CFG, ema=False),
        "abl_no_label_smoothing": dict(BEST_CFG, label_smoothing=0.0),
        "abl_no_dropout": dict(BEST_CFG, dropout=0.0),
    }
    if PREVIEW:
        variants = {k: variants[k] for k in
                    ("abl_no_augmentation", "abl_constant_lr", "abl_adam_instead_of_adamw")}
    for name, cfg in variants.items():
        r = train_run(name, cfg)
        record(r, "ablation")
        ablation.append(r)
    abl = pd.DataFrame([dict(best, name="SELECTED (all mechanisms)")] + ablation)
    abl = abl[["name", "val_macro_f1", "val_acc", "min_val_loss", "val_loss_roughness",
               "best_epoch", "epochs_trained", "train_macro_f1_at_best"]]
    abl.to_csv(OUT / "ablation_results.csv", index=False)
    print(abl.to_string(index=False))


## 11. Search figures

Saves the search and ablation figures.

In [ ]:
# ---------------------------------------------------------------- 11. Search figures
gs = pd.read_csv(OUT / "grid_search_results.csv")

a = gs[gs.stage == "A_architecture"].sort_values("val_macro_f1")
if len(a):
    fig, ax = plt.subplots(figsize=(10, max(3.5, 0.45 * len(a))))
    ax.barh(a["name"].str.replace("A_", "", regex=False), a["val_macro_f1"],
            color=["tab:orange" if str(bi).lower() == "true" else "tab:blue"
                   for bi in a["cfg_bidirectional"]])
    ax.set_xlim(a["val_macro_f1"].min() - 0.02, 1.0)
    ax.set_xlabel("validation macro-F1")
    ax.set_title("Stage A - architecture search (orange = BiLSTM, blue = LSTM)")
    for y, (v, p) in enumerate(zip(a["val_macro_f1"], a["params_total"])):
        ax.text(v + 0.001, y, f"{v:.4f} ({p/1e3:.0f}k params)", va="center", fontsize=8)
    fig.tight_layout()
    fig.savefig(FIG_DIR / "grid_search_architecture.png", dpi=150, bbox_inches="tight")
    plt.close(fig)

b = gs[gs.stage == "B_recipe"].sort_values("val_macro_f1")
if len(b):
    fig, ax = plt.subplots(figsize=(10, max(3.5, 0.45 * len(b))))
    ax.barh(b["name"].str.replace("B_", "", regex=False), b["val_macro_f1"], color="tab:green")
    ax.set_xlim(b["val_macro_f1"].min() - 0.02, 1.0)
    ax.set_xlabel("validation macro-F1")
    ax.set_title("Stage B - optimiser / LR / dropout search")
    for y, v in enumerate(b["val_macro_f1"]):
        ax.text(v + 0.001, y, f"{v:.4f}", va="center", fontsize=8)
    fig.tight_layout()
    fig.savefig(FIG_DIR / "grid_search_recipe.png", dpi=150, bbox_inches="tight")
    plt.close(fig)

if ablation:
    abl_s = pd.DataFrame(ablation).sort_values("val_macro_f1")
    fig, ax = plt.subplots(1, 2, figsize=(14, 4.5))
    ax[0].barh(abl_s["name"], abl_s["val_macro_f1"], color="tab:green")
    ax[0].set_xlim(abl_s["val_macro_f1"].min() - 0.02, 1.0)
    ax[0].set_title("Ablation - validation macro-F1")
    ax[1].barh(abl_s["name"], abl_s["val_loss_roughness"], color="tab:red")
    ax[1].set_title("Ablation - val-loss roughness (lower = smoother)")
    fig.tight_layout()
    fig.savefig(FIG_DIR / "ablation.png", dpi=150, bbox_inches="tight")
    plt.close(fig)


## 12. Final model and learning curves

Loads the selected checkpoint, writes `training_history_final.csv`, and plots the final learning curves and baseline-vs-tuned curves.

In [ ]:
# ---------------------------------------------------------------- 12. Final model and learning curves
final, ckpt = load_run_model(best)
final_cfg = ckpt["config"]
hist_final = pd.read_csv(Path(best["run_dir"]) / "history.csv")
hist_final.to_csv(OUT / "training_history_final.csv", index=False)
plot_history(hist_final, best["best_epoch"],
             f"Tuned {model_name_for(final_cfg).upper()} - learning curves ({best['name']})",
             FIG_DIR / "learning_curves_final.png", show=IN_COLAB)

base_run = pick_best(baselines) if baselines else None
if base_run is not None:
    hb = pd.read_csv(Path(base_run["run_dir"]) / "history.csv")
    hb.to_csv(OUT / "training_history_baseline.csv", index=False)
    fig, ax = plt.subplots(1, 2, figsize=(14, 4.5))
    ax[0].plot(hb.epoch, hb.val_loss,
               label=f"baseline {model_name_for(cfg_from(base_run))} "
                     f"(roughness {roughness(hb.val_loss):.3f})")
    ax[0].plot(hist_final.epoch, hist_final.val_loss,
               label=f"tuned (roughness {roughness(hist_final.val_loss):.3f})")
    ax[0].set_yscale("log")
    ax[0].set_title("Validation loss")
    ax[0].legend()
    ax[1].plot(hb.epoch, hb.val_macro_f1, label="baseline")
    ax[1].plot(hist_final.epoch, hist_final.val_macro_f1, label="tuned")
    ax[1].set_title("Validation macro-F1")
    ax[1].legend()
    for a_ in ax:
        a_.set_xlabel("epoch")
        a_.grid(alpha=.3)
    fig.tight_layout()
    fig.savefig(FIG_DIR / "baseline_vs_tuned_curves.png", dpi=150, bbox_inches="tight")
    if IN_COLAB:
        plt.show()
    plt.close(fig)


## 13. The single tuned test evaluation

The single tuned test evaluation: predictions, classification report, confusion matrices, per-subject accuracy and the two-panel figure.

In [ ]:
# ---------------------------------------------------------------- 13. THE single tuned test evaluation
P_tr = predict_proba(final, data["X_train"])
P_va = predict_proba(final, data["X_val"])
P_te = predict_proba(final, data["X_test"])
y_te, s_te = data["y_test"], data["subj_test"]
pred_te = P_te.argmax(1)

np.savez_compressed(FINAL_DIR / "predictions.npz",
                    P_train=P_tr, P_val=P_va, P_test=P_te, y_test=y_te, subj_test=s_te)

rep = classification_report(y_te, pred_te, labels=list(range(len(hc.CLASS_NAMES))),
                            target_names=hc.CLASS_NAMES, digits=4, zero_division=0)
print(rep)
open(OUT / "classification_report_test.txt", "w", encoding="utf-8").write(rep)

fig = hc.plot_confusion_matrix(y_te, pred_te, FIG_DIR / "confusion_matrix_test.png", "Test")
if IN_COLAB:
    plt.show()
plt.close(fig)
fig = hc.plot_confusion_matrix(data["y_val"], P_va.argmax(1),
                               FIG_DIR / "confusion_matrix_val.png", "Validation")
if IN_COLAB:
    plt.show()
plt.close(fig)

subj = hc.subject_stats(y_te, pred_te, s_te)["per_subject"]
pd.DataFrame({"subject": list(subj), "accuracy": list(subj.values())}) \
    .to_csv(OUT / "per_subject_accuracy_test.csv", index=False)

fig, ax = plt.subplots(1, 2, figsize=(14, 4.5))
ax[0].bar([str(s) for s in subj], list(subj.values()), color="tab:blue")
ax[0].axhline(np.mean(list(subj.values())), ls="--", c="k", label="mean")
ax[0].legend()
ax[0].set_ylim(max(0.0, min(subj.values()) - 0.05), 1.0)
ax[0].set_title("Test accuracy per subject")
ax[0].set_xlabel("subject")
ax[0].set_ylabel("accuracy")
f1s = [hc.compute_metrics(y_te, P_te)[f"f1_{c}"] for c in hc.CLASS_NAMES]
ax[1].bar(["WALK", "UP", "DOWN", "SIT", "STAND", "LAY"], f1s, color="tab:orange")
ax[1].set_ylim(max(0.0, min(f1s) - 0.05), 1.0)
ax[1].set_title("Test F1 per class")
for i, v in enumerate(f1s):
    ax[1].text(i, v + 0.003, f"{v:.3f}", ha="center", fontsize=8)
fig.tight_layout()
fig.savefig(FIG_DIR / "per_subject_and_per_class.png", dpi=150, bbox_inches="tight")
if IN_COLAB:
    plt.show()
plt.close(fig)


## 14. Save tuned model, efficiency and results row

Saves the tuned weights/config, efficiency numbers, and the `tuned` row in `lstm_results.csv`.

In [ ]:
# ---------------------------------------------------------------- 14. Save tuned model + efficiency + results row
tuned_name = model_name_for(final_cfg)
weights_path = FINAL_DIR / f"{tuned_name}_tuned_weights.pt"
torch.save(final.state_dict(), weights_path)

config_out = dict(
    final_cfg,
    model_name=tuned_name,
    preview=PREVIEW,
    selected_run=best["name"],
    input_shape=[128, 9],
    num_classes=len(hc.CLASS_NAMES),
    class_names=hc.CLASS_NAMES,
    dataset_version=meta["dataset_version"],
    owner=OWNER,
    best_epoch=best["best_epoch"],
    best_val_macro_f1=best["val_macro_f1"],
    hardware=hc.hardware_string(),
    framework=hc.framework_string(),
)
json.dump(config_out, open(FINAL_DIR / "config.json", "w"), indent=2, default=float)

efficiency = efficiency_for(final, weights_path, best)
training = training_for(best)
json.dump(efficiency | training, open(FINAL_DIR / "efficiency_metrics.json", "w"),
          indent=2, default=float)
print(json.dumps(efficiency | training, indent=2, default=float))

row_tuned = hc.save_results(
    OUT / "lstm_results.csv",
    model_name=tuned_name, family="RNN", owner=OWNER,
    input_representation="raw_9ch_128", run_tag="tuned",
    y_train=data["y_train"], proba_train=P_tr,
    y_val=data["y_val"], proba_val=P_va,
    y_test=y_te, proba_test=P_te, subj_test=s_te,
    efficiency=efficiency, training=training, seed=SEED,
    notes=notes_for(final_cfg, best, "tuned"), processed_dir=PROCESSED_DIR,
)


## 15. Baseline test rows and before/after table

Evaluates the fixed baselines once on test, writes their result rows, and builds `baseline_vs_tuned_test.csv`.

In [ ]:
# ---------------------------------------------------------------- 15. Baseline test rows + before/after table
baseline_rows = []
for r in baselines:
    model_b, ckpt_b = load_run_model(r)
    cfg_b = ckpt_b["config"]
    name_b = model_name_for(cfg_b)
    wpath_b = FINAL_DIR / f"{name_b}_baseline_weights.pt"
    torch.save(model_b.state_dict(), wpath_b)

    Pb_tr = predict_proba(model_b, data["X_train"])
    Pb_va = predict_proba(model_b, data["X_val"])
    Pb_te = predict_proba(model_b, data["X_test"])
    pred_b = Pb_te.argmax(1)

    rep_b = classification_report(y_te, pred_b, labels=list(range(len(hc.CLASS_NAMES))),
                                  target_names=hc.CLASS_NAMES, digits=4, zero_division=0)
    open(OUT / f"classification_report_{name_b}_baseline_test.txt", "w", encoding="utf-8").write(rep_b)

    eff_b = efficiency_for(model_b, wpath_b, r)
    row_b = hc.save_results(
        OUT / "lstm_results.csv",
        model_name=name_b, family="RNN", owner=OWNER,
        input_representation="raw_9ch_128", run_tag="baseline",
        y_train=data["y_train"], proba_train=Pb_tr,
        y_val=data["y_val"], proba_val=Pb_va,
        y_test=y_te, proba_test=Pb_te, subj_test=s_te,
        efficiency=eff_b, training=training_for(r), seed=SEED,
        notes=notes_for(cfg_b, r, "baseline"), processed_dir=PROCESSED_DIR,
    )
    baseline_rows.append(row_b)

base_row = max(baseline_rows, key=lambda x: x["val_macro_f1"]) if baseline_rows else None
if base_row is not None:
    metrics = ["test_accuracy", "test_macro_f1", "test_macro_precision", "test_macro_recall",
               "test_weighted_f1", "test_balanced_accuracy", "test_cohen_kappa", "test_mcc",
               "test_log_loss", "test_roc_auc_macro_ovr", "test_top2_accuracy",
               "sit_stand_confusion_rate", "walking_variants_confusion_rate",
               "static_dynamic_confusion_rate", "gap_train_test_f1", "params_total",
               "model_size_mb", "flops_per_window", "latency_ms_cpu_b1", "latency_ms_gpu_b1",
               "throughput_gpu_wps", "peak_gpu_mem_mb", "train_time_s", "epochs_trained",
               "best_epoch"]
    cmp = pd.DataFrame({
        "metric": metrics,
        "baseline": [base_row[m] for m in metrics],
        "tuned": [row_tuned[m] for m in metrics],
    })
    cmp["change"] = cmp["tuned"] - cmp["baseline"]
    cmp.to_csv(OUT / "baseline_vs_tuned_test.csv", index=False)
    print("\nBASELINE VS TUNED")
    print(cmp.to_string(index=False, float_format=lambda v: f"{v:.6g}"))


## 16. Full metrics and final checklist

Writes full metrics JSON, prints the comparison table and verifies every required file.

In [ ]:
# ---------------------------------------------------------------- 16. Full metrics + final checklist
extra = dict(
    preview=PREVIEW,
    selected_run=best["name"],
    selected_model=tuned_name,
    n_configs_searched=len(search_results),
    per_subject_test_accuracy=subj,
    full_train=hc.compute_metrics(data["y_train"], P_tr),
    full_val=hc.compute_metrics(data["y_val"], P_va),
    full_test=hc.compute_metrics(y_te, P_te),
    baseline_models=[r["model_name"] for r in baseline_rows],
    best_baseline=base_row["model_name"] if base_row is not None else None,
)
json.dump(extra, open(OUT / "metrics_full.json", "w"), indent=2, default=float)

results = pd.read_csv(OUT / "lstm_results.csv")
print("\nLSTM / BiLSTM RESULTS")
print(results[["model_name", "run_tag", "test_accuracy", "test_macro_f1", "test_mcc",
                "params_total", "latency_ms_cpu_b1", "flops_per_window"]]
      .sort_values(["run_tag", "test_macro_f1"], ascending=[True, False])
      .to_string(index=False))

required = [
    OUT / "lstm_results.csv",
    OUT / "training_history_final.csv",
    OUT / "training_history_baseline.csv",
    OUT / "classification_report_test.txt",
    OUT / "per_subject_accuracy_test.csv",
    OUT / "grid_search_results.csv",
    OUT / "baseline_vs_tuned_test.csv",
    OUT / "best_config.json",
    OUT / "ablation_results.csv",
    FIG_DIR / "learning_curves_final.png",
    FIG_DIR / "confusion_matrix_test.png",
    FIG_DIR / "confusion_matrix_val.png",
    FIG_DIR / "per_subject_and_per_class.png",
    FIG_DIR / "grid_search_architecture.png",
    FIG_DIR / "grid_search_recipe.png",
    FIG_DIR / "baseline_vs_tuned_curves.png",
    FIG_DIR / "ablation.png",
    FINAL_DIR / "predictions.npz",
    FINAL_DIR / f"{tuned_name}_tuned_weights.pt",
    FINAL_DIR / "config.json",
    FINAL_DIR / "efficiency_metrics.json",
    OUT / "metrics_full.json",
]
print("\nFILE CHECK")
for p in required:
    print(("OK      " if p.exists() else "MISSING "), p.relative_to(OUT))
print("\nAll outputs are in:", OUT)


## Hand-in checklist

After a full run, `lstm/` contains `lstm_results.csv` (baseline and tuned rows), the final and
baseline histories, the classification report, per-subject accuracy, the grid search, the
before/after table, `best_config.json`, `ablation_results.csv`, all required figures, and
`final_model/` with `predictions.npz`, the tuned weights, `config.json` and efficiency metrics.
